In [1]:

import numpy as np
import tensorflow as tf
import csv
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from scipy.optimize import minimize

In [2]:
data = []

with open('new_training_data.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)
    for row in reader:
        float_row = [float(item) for item in row[1:]]
        data.append(float_row)

data = np.array(data)
print(data[0,0:6],data[0,7])


[4.e-01 2.e+02 5.e+02 5.e-01 1.e-01 4.e-01] 98.75312743529798


In [3]:
param_bounds = [
    (0, 1),      # IR
    (50, 500),   # NG
    (50, 500),   # PS
    (0.1, 1),      # PC
    (0.1, 1),      # PM
    (0.4, 0.7)      # r1
]

def normalize_params(X, bounds):
    X_norm = np.empty_like(X)
    for i, (min_val, max_val) in enumerate(bounds):
        X_norm[:, i] = (X[:, i] - min_val) / (max_val - min_val)
    return X_norm

def denormalize_params(X_norm, bounds):
    X = np.empty_like(X_norm)
    for i, (min_val, max_val) in enumerate(bounds):
        X[:, i] = X_norm[:, i] * (max_val - min_val) + min_val
    return X

In [4]:

X = data[:, 0:6]  # Hyperparameters
y = data[:, 7:]  # Results

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
X_normalized = normalize_params(X, param_bounds)
print(X_normalized)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))
def buid_model():
    model = keras.Sequential([
        layers.Input(shape=(1,)),      
        layers.Dense(64, activation='relu'),  
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(6, activation='sigmoid')  # 6 outputs = hyperparameters (normalized 0–1)
    ])


    model.compile(optimizer='adam', loss='mse')
    return model
model = buid_model()    

[[0.4        0.33333333 1.         0.44444444 0.         0.        ]
 [0.4        0.33333333 1.         0.44444444 0.         0.        ]
 [0.4        0.33333333 1.         0.44444444 0.         0.        ]
 ...
 [0.9        0.33333333 1.         0.77777778 0.22222222 1.        ]
 [0.9        0.33333333 1.         0.77777778 0.22222222 1.        ]
 [0.9        0.33333333 1.         0.77777778 0.22222222 1.        ]]


In [5]:
model.fit(y_scaled, X_normalized,epochs=500,validation_split=0.2, verbose=0)

In [6]:
import jpype
import jpype.imports

def postprocess_params(params):
    # Ensure it's a clean copy and 1D
    params = params.flatten().copy()

    # Process each parameter individually
    ps = int(round(params[2]))
    ps = max(50, min(ps, 500))
    if ps % 2 != 0:
        ps += 1 if ps < 500 else -1

    nmp = params[5]
    nmp = max(0.3, min(nmp, 0.75))

    ng = int(round(params[1]))
    ng = max(50, min(ng, 500))

    ir = float(np.clip(params[0], 0, 1))
    pc = float(np.clip(params[3], 0, 1))
    pm = float(np.clip(params[4], 0, 1))

    numeric_params = np.array([ir, ng, ps, pc, pm, nmp])

    # String version for Java input
    java_params = [
        f"{ir:.6f}",  # float
        str(ng),      # int
        str(ps),      # int
        f"{pc:.6f}",  # float
        f"{pm:.6f}",
          f"{nmp:.6f}"  # float
    ]

    return numeric_params, java_params

def run_java_algorithm(params):
    if not jpype.isJVMStarted():
        jpype.startJVM(classpath=["C:/Users/USER/Desktop/my_projects/optimization_with_java/bin"])
    GGA = jpype.JClass("GGA.GGA")  # Just the class name
    java_params = jpype.JArray(jpype.JString)([str(p) for p in params])
    print("→ Running Java GGA with params:", java_params)
    return GGA.tryData(java_params)

def objective(scaled_result_input):
    # Reshape input (1 feature)
    scaled_result_input = np.array(scaled_result_input).reshape(1, -1)
    
    # Predict normalized hyperparameters
    predicted_params_norm = model.predict(scaled_result_input, verbose=0)

    # Denormalize to real parameter space
    predicted_params = denormalize_params(predicted_params_norm, param_bounds)

    # Optionally postprocess (round integers, etc.)
    predicted_params ,j= postprocess_params(predicted_params)

    # Evaluate GGA algorithm on MKP using these predicted parameters
     # Get numeric params only
    score = run_java_algorithm(j)  # You define this

    return -score  # Negate if you want to maximize


# Start from mean result (scaled)
initial_result = scaler_y.transform(np.mean(y).reshape(1, -1))[0]

# Bounds for input result ∈ scaled space (roughly [-2, 2] for standardized data)
bounds = [(-2, 2)]  # only one input dimension now

# Run optimization
result = minimize(objective, initial_result, bounds=bounds, method='L-BFGS-B',options={
        'maxiter': 5,
        'disp': True,       # Show optimization info
        'gtol': 1e-6,        # Stop if gradient norm < 1e-6
    })


→ Running Java GGA with params: ['0.593552', '202', '496', '0.584144', '0.205952', '0.591403']
→ Running Java GGA with params: ['0.593552', '202', '496', '0.584144', '0.205952', '0.591403']


In [7]:
best_result=-result.fun
# Best result input (scaled)
best_result_scaled = scaler_y.transform(np.array([[best_result]]))
# Predict best hyperparameters from it
best_params_norm = model.predict(best_result_scaled, verbose=0)
best_params = denormalize_params(best_params_norm, param_bounds)
best_params = postprocess_params(best_params)
print(result)


print("✅ Best parameters found (original scale):", best_params)
print("initial_result", initial_result)
print("🎯 best result", best_result)

  message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
  success: True
   status: 0
      fun: -99.0
        x: [ 0.000e+00]
      nit: 0
      jac: [ 0.000e+00]
     nfev: 2
     njev: 1
 hess_inv: <1x1 LbfgsInvHessProduct with dtype=float64>
✅ Best parameters found (original scale): (array([5.38966596e-01, 2.02000000e+02, 4.96000000e+02, 6.57109499e-01,
       1.70856014e-01, 5.16985297e-01]), ['0.538967', '202', '496', '0.657109', '0.170856', '0.516985'])
initial_result [0.]
🎯 best result 99.0
